# setup

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["PYTHONWARNINGS"] = "ignore"

!pip install -q ultralytics

import torch
from ultralytics import YOLO

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
2.10.0+cu128
True
Tesla T4


# DATA

In [2]:
!wget -q --show-progress -O rpee_heads.zip "http://ped.fz-juelich.de/data/machine_learning/2024_11_Recognition_In_Field_Studies/data/2024rpee_heads_dataset.zip"

!unzip -q rpee_heads.zip -d rpee_heads

rpee_heads.zip      100%[===================>]   1.08G  14.1MB/s    in 74s     


In [3]:
!unzip -oq rpee_heads.zip -d rpee_heads

import os

for root, dirs, files in os.walk("rpee_heads"):
    level = root.replace("rpee_heads", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... {len(files)} files total")

rpee_heads/
  training/
    train/
      images/
      labels/
  validation/
    val/
      images/
      labels/
  testing/
    test/
      images/
      labels/


In [4]:
import os

base = "rpee_heads"
splits = {
    "train": "training/train",
    "val": "validation/val",
    "test": "testing/test"
}

for name, path in splits.items():
    img_dir = os.path.join(base, path, "images")
    lbl_dir = os.path.join(base, path, "labels")
    n_images = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    n_labels = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(name, "images:", n_images, "labels:", n_labels)

sample_label_dir = os.path.join(base, splits["train"], "labels")
sample_file = os.listdir(sample_label_dir)[0]
with open(os.path.join(sample_label_dir, sample_file)) as f:
    content = f.read()
print(sample_file)
print(content[:300])

train images: 1346 labels: 1346
val images: 246 labels: 246
test images: 294 labels: 294
3C080_entry1_bend90_wI_nM_corrected_trimmed_1_frames_3C080_entry1_bend90_wI_nM_corrected_trimmed_1_frames_frame_2500.txt
0 0.367992 0.420290 0.018349 0.021739
0 0.604485 0.519928 0.016310 0.025362
0 0.596330 0.440217 0.014271 0.021739
0 0.310398 0.415761 0.017329 0.023551
0 0.494393 0.293478 0.018349 0.021739
0 0.426096 0.451087 0.018349 0.025362
0 0.142712 0.381341 0.014271 0.019928
0 0.711009 0.623188 0.015291 0.018


# data.yaml Config

In [5]:
yaml_content = """path: /kaggle/working/rpee_heads
train: training/train/images
val: validation/val/images
test: testing/test/images

names:
  0: head
"""

with open("/kaggle/working/rpee_heads.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

path: /kaggle/working/rpee_heads
train: training/train/images
val: validation/val/images
test: testing/test/images

names:
  0: head



#  Model Setup and Training

In [6]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")

results = model.train(
    data="/kaggle/working/rpee_heads.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/kaggle/working/runs",
    name="rpee_heads_yolo26n",
    patience=15
)

Ultralytics 8.4.90 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rpee_heads.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rpee_heads_yolo26n, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=Tru

#  Evaluation on Test Set and Visual Check


In [7]:
from ultralytics import YOLO

best_model = YOLO("/kaggle/working/runs/rpee_heads_yolo26n/weights/best.pt")

test_results = best_model.predict(
    source="/kaggle/working/rpee_heads/testing/test/images",
    conf=0.3,
    save=True,
    project="/kaggle/working/runs",
    name="test_predictions"
)

print(len(test_results), "images processed")


image 1/294 /kaggle/working/rpee_heads/testing/test/images/2C100_entry3_straight_hM_corrected_trimmed_frames_2C100_entry3_straight_hM_corrected_trimmed_frames_frame_5000.jpg: 384x640 94 heads, 55.1ms
image 2/294 /kaggle/working/rpee_heads/testing/test/images/2C100_entry3_straight_hM_corrected_trimmed_frames_2C100_entry3_straight_hM_corrected_trimmed_frames_frame_5250.jpg: 384x640 78 heads, 13.7ms
image 3/294 /kaggle/working/rpee_heads/testing/test/images/2C100_entry3_straight_hM_corrected_trimmed_frames_2C100_entry3_straight_hM_corrected_trimmed_frames_frame_5500.jpg: 384x640 62 heads, 12.4ms
image 4/294 /kaggle/working/rpee_heads/testing/test/images/2C100_entry3_straight_hM_corrected_trimmed_frames_2C100_entry3_straight_hM_corrected_trimmed_frames_frame_5750.jpg: 384x640 57 heads, 9.3ms
image 5/294 /kaggle/working/rpee_heads/testing/test/images/3C080_entry1_bend90_wI_nM_corrected_trimmed_1_frames_3C080_entry1_bend90_wI_nM_corrected_trimmed_1_frames_frame_12000.jpg: 384x640 87 heads, 

In [8]:
import shutil
import os

os.makedirs("/kaggle/working/export", exist_ok=True)

shutil.copy(
    "/kaggle/working/runs/rpee_heads_yolo26n/weights/best.pt",
    "/kaggle/working/export/head_detector_yolo26n.pt"
)

shutil.copy(
    "/kaggle/working/rpee_heads.yaml",
    "/kaggle/working/export/rpee_heads.yaml"
)

with open("/kaggle/working/export/model_info.txt", "w") as f:
    f.write("Model: YOLO26n fine-tuned on RPEE-Heads\n")
    f.write("Classes: 0=head\n")
    f.write("Input size: 640x640\n")
    f.write("Precision: 0.874\n")
    f.write("Recall: 0.729\n")
    f.write("mAP50: 0.809\n")
    f.write("mAP50-95: 0.443\n")
    f.write("Usage: model = YOLO('head_detector_yolo26n.pt')\n")

print(os.listdir("/kaggle/working/export"))

['head_detector_yolo26n.pt', 'rpee_heads.yaml', 'model_info.txt']
